In [1]:
import os
import pandas as pd

# 1. Load Adedamola's raw FX file (semicolon-separated)
df = pd.read_csv("../data/raw/cbn_exchange_rate_raw.csv", sep=";")

# 2. Standardize dates to YYYY-MM-01 and ensure Close price is numeric
df["date"] = pd.to_datetime(df["Date"]).dt.to_period("M").dt.to_timestamp()
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

# 3. Compute monthly average exchange rate across 2020-01-01 to 2025-12-01
monthly_fx = (
    df.groupby("date")["Close"]
    .mean()
    .round(2)
    .rename("usd_ngn_rate")
    .reset_index()
)

# 4. Verify full 72-month alignment
full_months = pd.DataFrame({"date": pd.date_range("2020-01-01", "2025-12-01", freq="MS")})
fx_clean = full_months.merge(monthly_fx, on="date", how="left")

# 5. Save cleaned monthly FX file
os.makedirs("../data/processed", exist_ok=True)
fx_clean.to_csv("../data/processed/clean_fx_rates.csv", index=False)

print("Done! Saved data/processed/clean_fx_rates.csv")
print("Total rows:", len(fx_clean), "| Missing values:", fx_clean["usd_ngn_rate"].isnull().sum())
fx_clean.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/cbn_exchange_rate_raw.csv'

In [2]:
import os
import glob
import pandas as pd

# 1. Automatically locate Adedamola's FX file regardless of notebook folder or filename
candidates = (
    glob.glob("../data/raw/*fx*") +
    glob.glob("../data/raw/*cbn*") +
    glob.glob("data/raw/*fx*") +
    glob.glob("data/raw/*cbn*")
)

if not candidates:
    raise FileNotFoundError(
        "Could not find the FX file in data/raw/. Copy 'fx_2020-2025.csv.csv.xls' into Lagos_price_pulse/data/raw/ first."
    )

raw_path = candidates[0]
print("Found file at:", raw_path)

# 2. Load semicolon-separated FX data
df = pd.read_csv(raw_path, sep=";")

# 3. Standardize dates to YYYY-MM-01 and ensure Close price is numeric
df["date"] = pd.to_datetime(df["Date"]).dt.to_period("M").dt.to_timestamp()
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

# 4. Compute monthly average exchange rate across 2020-01-01 to 2025-12-01
monthly_fx = (
    df.groupby("date")["Close"]
    .mean()
    .round(2)
    .rename("usd_ngn_rate")
    .reset_index()
)

# 5. Verify full 72-month alignment
full_months = pd.DataFrame({"date": pd.date_range("2020-01-01", "2025-12-01", freq="MS")})
fx_clean = full_months.merge(monthly_fx, on="date", how="left")

# 6. Save cleaned monthly FX file to the right processed/ path
out_dir = "../data/processed" if os.path.exists("../data") else "data/processed"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "clean_fx_rates.csv")
fx_clean.to_csv(out_path, index=False)

print("Done! Saved:", out_path)
print("Total rows:", len(fx_clean), "| Missing values:", fx_clean["usd_ngn_rate"].isnull().sum())
fx_clean.head(5)

Found file at: ../data/raw\cbn_exchange_rate_raw.csv.xls
Done! Saved: ../data/processed\clean_fx_rates.csv
Total rows: 72 | Missing values: 0


,date,usd_ngn_rate
0,2020-01-01,359.67
1,2020-02-01,361.53
2,2020-03-01,363.85
3,2020-04-01,383.74
4,2020-05-01,388.72
